In [ ]:
#cnn with original dataset Confidence Interval (Bootstrap)
import pandas as pd
import numpy as np
from scipy import stats

# =========================
# Create DataFrame from your runs
# =========================
data = {
    "Val_Accuracy": [0.992860,0.991456,0.982678,0.995318,0.993797,0.993095,0.961493,0.995435,0.994265,0.994382],
    "Val_Loss": [0.027901,0.041532,0.065112,0.019994,0.022228,0.025715,0.304745,0.019365,0.022638,0.024168],
    "MCC": [0.992629,0.991181,0.982135,0.995166,0.993597,0.992873,0.960954,0.995287,0.994079,0.994199],
    "ROC_AUC": [0.999979,0.999923,0.999920,0.999988,0.999974,0.999974,0.998694,0.999991,0.999985,0.999960],
    "F1_Weighted": [0.992856,0.991446,0.982659,0.995318,0.993788,0.993089,0.960867,0.995438,0.994264,0.994380],
    "Precision_Weighted": [0.992925,0.991586,0.983375,0.995341,0.993875,0.993227,0.977632,0.995479,0.994326,0.994408],
    "Recall_Weighted": [0.992860,0.991456,0.982678,0.995318,0.993797,0.993095,0.961493,0.995435,0.994265,0.994382],
    "Training_Time(s)": [720.699108,684.075694,718.841711,665.623284,644.432933,706.582799,640.895652,579.559939,747.852611,685.929550],
    "Memory_Used(MB)": [250.007812,269.785156,204.382812,200.421875,262.765625,133.468750,170.492188,268.550781,132.437500,249.218750]
}

df = pd.DataFrame(data)

# =========================
# CI settings
# =========================
metrics = df.columns
n = len(df)
t_value = stats.t.ppf(1 - 0.025, df=n-1)

results = []

for col in metrics:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1)

    se = std / np.sqrt(n)
    ci_low = mean - t_value * se
    ci_high = mean + t_value * se

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

print(result_df)

# save
result_df.to_excel("CNN_final_results_with_CI.xlsx", index=False)

               Metric               Mean ± SD                    95% CI
0        Val_Accuracy     0.989478 ± 0.010504      [0.981964, 0.996992]
1            Val_Loss     0.057340 ± 0.088041     [-0.005641, 0.120320]
2                 MCC     0.989210 ± 0.010634      [0.981603, 0.996817]
3             ROC_AUC     0.999839 ± 0.000403      [0.999550, 1.000127]
4         F1_Weighted     0.989410 ± 0.010690      [0.981763, 0.997058]
5  Precision_Weighted     0.991217 ± 0.005918      [0.986984, 0.995451]
6     Recall_Weighted     0.989478 ± 0.010504      [0.981964, 0.996992]
7    Training_Time(s)  679.449328 ± 49.005123  [644.393175, 714.505481]
8     Memory_Used(MB)  214.153125 ± 54.067915  [175.475268, 252.830981]


In [ ]:
import pandas as pd
import numpy as np

# =========================
# Data (10 runs)
# =========================
df = pd.DataFrame({
    "Val_Accuracy": [0.992860,0.991456,0.982678,0.995318,0.993797,0.993095,0.961493,0.995435,0.994265,0.994382],
    "Val_Loss": [0.027901,0.041532,0.065112,0.019994,0.022228,0.025715,0.304745,0.019365,0.022638,0.024168],
    "MCC": [0.992629,0.991181,0.982135,0.995166,0.993597,0.992873,0.960954,0.995287,0.994079,0.994199],
    "ROC_AUC": [0.999979,0.999923,0.999920,0.999988,0.999974,0.999974,0.998694,0.999991,0.999985,0.999960],
    "F1_Weighted": [0.992856,0.991446,0.982659,0.995318,0.993788,0.993089,0.960867,0.995438,0.994264,0.994380],
    "Precision_Weighted": [0.992925,0.991586,0.983375,0.995341,0.993875,0.993227,0.977632,0.995479,0.994326,0.994408],
    "Recall_Weighted": [0.992860,0.991456,0.982678,0.995318,0.993797,0.993095,0.961493,0.995435,0.994265,0.994382],
    "Training_Time": [720.699108,684.075694,718.841711,665.623284,644.432933,706.582799,640.895652,579.559939,747.852611,685.929550],
    "Memory_Used": [250.007812,269.785156,204.382812,200.421875,262.765625,133.468750,170.492188,268.550781,132.437500,249.218750]
})

# =========================
# Bootstrap CI (5000 resamples)
# =========================
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, 100 * (alpha/2))
    upper = np.percentile(boot_means, 100 * (1 - alpha/2))

    return lower, upper

# =========================
# Compute results
# =========================
results = []

for col in df.columns:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1)
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

print(result_df)

# save
result_df.to_excel("CNN_bootstrap_CI_5000.xlsx", index=False)

               Metric               Mean ± SD          Bootstrap 95% CI
0        Val_Accuracy     0.989478 ± 0.010504      [0.982420, 0.994090]
1            Val_Loss     0.057340 ± 0.088041      [0.023912, 0.115185]
2                 MCC     0.989210 ± 0.010634      [0.982283, 0.993934]
3             ROC_AUC     0.999839 ± 0.000403      [0.999577, 0.999977]
4         F1_Weighted     0.989410 ± 0.010690      [0.982473, 0.994098]
5  Precision_Weighted     0.991217 ± 0.005918      [0.987307, 0.994170]
6     Recall_Weighted     0.989478 ± 0.010504      [0.982560, 0.994113]
7       Training_Time  679.449328 ± 49.005123  [649.526871, 706.541902]
8         Memory_Used  214.153125 ± 54.067915  [181.977598, 245.197666]


In [ ]:
import pandas as pd
import numpy as np

# =========================
# CNN with undersampling
# =========================
df = pd.DataFrame({
    "Val_Accuracy": [0.990352, 0.989107, 0.992375, 0.991597, 0.852163, 0.989574, 0.988017, 0.990196, 0.978992, 0.985839],
    "Val_Loss": [0.046143, 0.045229, 0.032652, 0.033725, 0.925336, 0.043928, 0.046303, 0.046465, 0.081366, 0.055768],
    "MCC": [0.990046, 0.988759, 0.992133, 0.991329, 0.851945, 0.989242, 0.987648, 0.989884, 0.978359, 0.985399],
    "ROC_AUC": [0.999818, 0.999881, 0.999957, 0.999927, 0.997454, 0.999874, 0.999940, 0.999903, 0.999861, 0.999913],
    "F1_Weighted": [0.990363, 0.989089, 0.992383, 0.991604, 0.855370, 0.989567, 0.988068, 0.990218, 0.979032, 0.985821],
    "Precision_Weighted": [0.990550, 0.989171, 0.992510, 0.991715, 0.928994, 0.989716, 0.988618, 0.990374, 0.980405, 0.986313],
    "Recall_Weighted": [0.990352, 0.989107, 0.992375, 0.991597, 0.852163, 0.989574, 0.988017, 0.990196, 0.978992, 0.985839],
    "Training_Time": [443.773410, 404.334795, 392.731281, 406.645728, 402.517840, 398.804440, 388.840323, 397.213548, 376.466319, 390.839134],
    "Memory_Used": [1151.257812, 281.800781, 250.011719, 217.335938, 182.640625, 183.312500, 245.109375, 154.218750, 185.769531, 239.421875]
})

# =========================
# Bootstrap CI (5000 resamples)
# =========================
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, 100 * (alpha/2))
    upper = np.percentile(boot_means, 100 * (1 - alpha/2))

    return lower, upper

# =========================
# Compute results
# =========================
results = []

for col in df.columns:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1)
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

print(result_df)

# save
result_df.to_excel("CNN_bootstrap_CI_5000.xlsx", index=False)

               Metric                Mean ± SD          Bootstrap 95% CI
0        Val_Accuracy      0.974821 ± 0.043265      [0.946809, 0.990087]
1            Val_Loss      0.135692 ± 0.277782      [0.042318, 0.313688]
2                 MCC      0.974474 ± 0.043230      [0.946486, 0.989727]
3             ROC_AUC      0.999653 ± 0.000774      [0.999155, 0.999916]
4         F1_Weighted      0.975151 ± 0.042258      [0.947917, 0.990066]
5  Precision_Weighted      0.982837 ± 0.019226      [0.970065, 0.990335]
6     Recall_Weighted      0.974821 ± 0.043265      [0.946670, 0.990056]
7       Training_Time   400.216682 ± 17.667483  [391.247243, 412.240214]
8         Memory_Used  309.087891 ± 298.486927  [197.218252, 504.533740]


In [ ]:
import pandas as pd
import numpy as np

# =========================
# CNN with Oversampling
# =========================
df = pd.DataFrame({
    "Val_Accuracy": [0.996128, 0.997118, 0.996037, 0.991895, 0.997388, 0.996668, 0.993696, 0.996128, 0.994777, 0.886401],
    "Val_Loss": [0.013219, 0.012121, 0.019859, 0.039833, 0.012697, 0.012199, 0.026438, 0.018536, 0.021927, 0.044160],
    "MCC": [0.996004, 0.997026, 0.995912, 0.991640, 0.997305, 0.996564, 0.993497, 0.996003, 0.994610, 0.985981],
    "ROC_AUC": [0.999986, 0.999936, 0.999968, 0.999893, 0.999958, 0.999959, 0.999931, 0.999925, 0.999917, 0.999932],
    "F1_Weighted": [0.996123, 0.997115, 0.996037, 0.991908, 0.997388, 0.996666, 0.993694, 0.996126, 0.994779, 0.986478],
    "Precision_Weighted": [0.996157, 0.997149, 0.996098, 0.992128, 0.997415, 0.996772, 0.993820, 0.996141, 0.994834, 0.987083],
    "Recall_Weighted": [0.996128, 0.997118, 0.996037, 0.991895, 0.997388, 0.996668, 0.993696, 0.996128, 0.994777, 0.986401],
    "Training_Time": [783.916794, 779.279436, 723.924432, 797.485343, 669.204534, 783.150548, 669.662425, 707.477385, 723.238286, 707.622088],
    "Memory_Used": [1088.898438, 203.113281, 164.882812, 162.015625, 131.667969, 142.511719, 232.265625, 96.441406, 142.953125, 216.519531]
})

# =========================
# Bootstrap CI (5000 resamples)
# =========================
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, 100 * (alpha/2))
    upper = np.percentile(boot_means, 100 * (1 - alpha/2))

    return lower, upper

# =========================
# Compute results
# =========================
results = []

for col in df.columns:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1)
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

print(result_df)

# save
result_df.to_excel("CNN_bootstrap_CI_5000.xlsx", index=False)

               Metric                Mean ± SD          Bootstrap 95% CI
0        Val_Accuracy      0.984624 ± 0.034553      [0.962498, 0.996335]
1            Val_Loss      0.022099 ± 0.011559      [0.015890, 0.029417]
2                 MCC      0.994454 ± 0.003444      [0.992143, 0.996172]
3             ROC_AUC      0.999940 ± 0.000027      [0.999924, 0.999956]
4         F1_Weighted      0.994631 ± 0.003319      [0.992505, 0.996287]
5  Precision_Weighted      0.994760 ± 0.003148      [0.992684, 0.996351]
6     Recall_Weighted      0.994624 ± 0.003342      [0.992498, 0.996299]
7       Training_Time   734.496127 ± 48.235497  [705.971888, 763.302354]
8         Memory_Used  258.126953 ± 294.826802  [146.647998, 450.072588]


In [ ]:
import pandas as pd
import numpy as np

# =========================
# Data (10 runs) - Updated with latest results
# =========================
df = pd.DataFrame({
    "Val_Accuracy": [0.984399, 0.968799, 0.987520, 0.981279, 0.967239, 0.979719, 0.976599, 0.976599, 0.973479, 0.973479],
    "Val_Loss": [0.082863, 0.132471, 0.101366, 0.083952, 0.116621, 0.109247, 0.097640, 0.108185, 0.122056, 0.139102],
    "MCC": [0.983908, 0.967869, 0.987137, 0.980722, 0.966285, 0.979089, 0.975876, 0.975910, 0.972692, 0.972790],
    "ROC_AUC": [0.999725, 0.999695, 0.999253, 0.999785, 0.999638, 0.999331, 0.999580, 0.999435, 0.999577, 0.999277],
    "F1_Weighted": [0.984390, 0.968636, 0.987521, 0.981156, 0.967075, 0.979732, 0.976650, 0.976385, 0.973204, 0.974070],
    "Precision_Weighted": [0.985030, 0.971534, 0.988373, 0.982886, 0.970925, 0.980888, 0.978097, 0.978838, 0.975664, 0.978889],
    "Recall_Weighted": [0.984399, 0.968799, 0.987520, 0.981279, 0.967239, 0.979719, 0.976599, 0.976599, 0.973479, 0.973479],
    "Training_Time": [726.152705, 700.211145, 711.465278, 708.685914, 717.989220, 713.670773, 706.363833, 729.968301, 726.327500, 734.649886],
    "Memory_Used": [-476.949219, 93.550781, 43.527344, 194.335938, 46.031250, 41.101562, 288.105469, -165.242188, 210.371094, -123.375000]
})

# =========================
# CNN_LSTM with original dataset
# =========================
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, 100 * (alpha/2))
    upper = np.percentile(boot_means, 100 * (1 - alpha/2))

    return lower, upper

# =========================
# Compute results
# =========================
results = []

for col in df.columns:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1)
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

print(result_df.to_string(index=False))

# save
result_df.to_excel("CNN_bootstrap_CI_5000.xlsx", index=False)

            Metric              Mean ± SD          Bootstrap 95% CI
      Val_Accuracy    0.976911 ± 0.006487      [0.973167, 0.980811]
          Val_Loss    0.109350 ± 0.018806      [0.098640, 0.120071]
               MCC    0.976228 ± 0.006664      [0.972380, 0.980089]
           ROC_AUC    0.999530 ± 0.000193      [0.999415, 0.999637]
       F1_Weighted    0.976882 ± 0.006514      [0.973189, 0.980694]
Precision_Weighted    0.979112 ± 0.005535      [0.975962, 0.982433]
   Recall_Weighted    0.976911 ± 0.006487      [0.973167, 0.980659]
     Training_Time 717.548456 ± 11.324873  [710.969784, 724.249813]
       Memory_Used 15.145703 ± 222.908754 [-123.187696, 137.000283]


In [ ]:
import pandas as pd
import numpy as np

# =========================
# Data (10 runs)
# =========================
df = pd.DataFrame({
    "Val_Accuracy": [0.984399, 0.968799, 0.987520, 0.981279, 0.967239, 0.979719, 0.976599, 0.976599, 0.973479, 0.973479],
    "Val_Loss": [0.082863, 0.132471, 0.101366, 0.083952, 0.116621, 0.109247, 0.097640, 0.108185, 0.122056, 0.139102],
    "MCC": [0.983908, 0.967869, 0.987137, 0.980722, 0.966285, 0.979089, 0.975876, 0.975910, 0.972692, 0.972790],
    "ROC_AUC": [0.999725, 0.999695, 0.999253, 0.999785, 0.999638, 0.999331, 0.999580, 0.999435, 0.999577, 0.999277],
    "F1_Weighted": [0.984390, 0.968636, 0.987521, 0.981156, 0.967075, 0.979732, 0.976650, 0.976385, 0.973204, 0.974070],
    "Precision_Weighted": [0.985030, 0.971534, 0.988373, 0.982886, 0.970925, 0.980888, 0.978097, 0.978838, 0.975664, 0.978889],
    "Recall_Weighted": [0.984399, 0.968799, 0.987520, 0.981279, 0.967239, 0.979719, 0.976599, 0.976599, 0.973479, 0.973479],
    "Training_Time": [726.152705, 700.211145, 711.465278, 708.685914, 717.989220, 713.670773, 706.363833, 729.968301, 726.327500, 734.649886],
    "Memory_Used": [-476.949219, 93.550781, 43.527344, 194.335938, 46.031250, 41.101562, 288.105469, -165.242188, 210.371094, -123.375000]
})

# =========================
# FIX MEMORY ISSUE (IMPORTANT)
# =========================
df["Memory_Used"] = df["Memory_Used"].abs()

# =========================
# Bootstrap CI (5000 resamples)
# =========================
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, 100 * (alpha / 2))
    upper = np.percentile(boot_means, 100 * (1 - alpha / 2))

    return lower, upper

# =========================
# Compute results
# =========================
results = []

for col in df.columns:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1)
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

print(result_df.to_string(index=False))

# save
result_df.to_excel("CNN_bootstrap_CI_5000.xlsx", index=False)

            Metric               Mean ± SD         Bootstrap 95% CI
      Val_Accuracy     0.976911 ± 0.006487     [0.973167, 0.980811]
          Val_Loss     0.109350 ± 0.018806     [0.098411, 0.120319]
               MCC     0.976228 ± 0.006664     [0.972375, 0.980388]
           ROC_AUC     0.999530 ± 0.000193     [0.999415, 0.999642]
       F1_Weighted     0.976882 ± 0.006514     [0.973033, 0.980850]
Precision_Weighted     0.979112 ± 0.005535     [0.975927, 0.982283]
   Recall_Weighted     0.976911 ± 0.006487     [0.973167, 0.980655]
     Training_Time  717.548456 ± 11.324873 [710.997855, 724.187173]
       Memory_Used 168.258985 ± 135.964815  [96.882285, 252.785283]


In [ ]:
import pandas as pd
import numpy as np

# =========================
# CNN_LSTM with undersamplng
# =========================
df = pd.DataFrame({
    "Val_Accuracy": [0.995851, 0.995851, 0.995851, 0.993776, 0.995851, 0.995851, 0.995851, 0.995851, 0.995851, 0.995851],
    "Val_Loss": [0.019020, 0.014143, 0.016820, 0.033855, 0.019125, 0.020543, 0.011627, 0.016832, 0.014921, 0.020702],
    "MCC": [0.995726, 0.995726, 0.995726, 0.993588, 0.995726, 0.995726, 0.995726, 0.995726, 0.995726, 0.995726],
    "ROC_AUC": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    "F1_Weighted": [0.995846, 0.995846, 0.995846, 0.993767, 0.995846, 0.995846, 0.995846, 0.995846, 0.995846, 0.995846],
    "Precision_Weighted": [0.996110, 0.996110, 0.996110, 0.994157, 0.996110, 0.996110, 0.996110, 0.996110, 0.996110, 0.996110],
    "Recall_Weighted": [0.995851, 0.995851, 0.995851, 0.993776, 0.995851, 0.995851, 0.995851, 0.995851, 0.995851, 0.995851],
    "Training_Time": [497.765136, 510.435865, 491.247386, 494.175308, 531.017695, 514.278084, 502.165158, 500.531753, 504.917428, 488.511259],
    "Memory_Used": [224.101562, 264.925781, 80.250000, 233.605469, 46.300781, 59.695312, 257.679688, 127.167969, 79.750000, 236.667969]
})

# =========================
# Bootstrap CI (5000 resamples)
# =========================
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, 100 * (alpha/2))
    upper = np.percentile(boot_means, 100 * (1 - alpha/2))

    return lower, upper

# =========================
# Compute results
# =========================
results = []

for col in df.columns:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1)
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

print(result_df.to_string(index=False))

# save
result_df.to_excel("CNN_bootstrap_CI_5000.xlsx", index=False)

            Metric              Mean ± SD         Bootstrap 95% CI
      Val_Accuracy    0.995644 ± 0.000656     [0.995228, 0.995851]
          Val_Loss    0.018759 ± 0.006050     [0.015690, 0.022776]
               MCC    0.995512 ± 0.000676     [0.995085, 0.995726]
           ROC_AUC    1.000000 ± 0.000000     [1.000000, 1.000000]
       F1_Weighted    0.995638 ± 0.000657     [0.995222, 0.995846]
Precision_Weighted    0.995915 ± 0.000618     [0.995524, 0.996110]
   Recall_Weighted    0.995644 ± 0.000656     [0.995228, 0.995851]
     Training_Time 503.504507 ± 12.599495 [496.888782, 511.350853]
       Memory_Used 161.014453 ± 89.945547 [106.793584, 211.455801]


MobileNetV2

In [ ]:
import pandas as pd
import numpy as np

# =========================
# MobilenetV2 with original
# =========================
df = pd.DataFrame({
    "Val_Accuracy": [0.976826, 0.977645, 0.978464, 0.976592, 0.985019, 0.978464, 0.981156, 0.978230, 0.978581, 0.977060],
    "Val_Loss": [0.069414, 0.070262, 0.066109, 0.069250, 0.049221, 0.067706, 0.059518, 0.068895, 0.064357, 0.072770],
    "MCC": [0.976081, 0.976925, 0.977772, 0.975834, 0.984533, 0.977770, 0.980545, 0.977527, 0.977891, 0.976319],
    "ROC_AUC": [0.999882, 0.999877, 0.999900, 0.999895, 0.999921, 0.999900, 0.999913, 0.999884, 0.999915, 0.999881],
    "F1_Weighted": [0.976841, 0.977641, 0.978475, 0.976584, 0.985009, 0.978467, 0.981153, 0.978250, 0.978586, 0.977065],
    "Precision_Weighted": [0.977261, 0.978006, 0.978849, 0.976833, 0.985136, 0.978771, 0.981329, 0.978566, 0.978882, 0.977389],
    "Recall_Weighted": [0.976826, 0.977645, 0.978464, 0.976592, 0.985019, 0.978464, 0.981156, 0.978230, 0.978581, 0.977060],
    "Training_Time": [1093.022180, 988.173548, 1011.535979, 1081.126071, 1054.128004, 1069.201602, 1169.219979, 1215.669739, 1149.923759, 1227.439125],
    "Memory_Used": [1312.269531, 468.617188, 376.347656, 336.820312, 398.992188, 320.601562, 382.089844, 292.386719, 313.808594, 314.085938]
})

# =========================
# Bootstrap CI (5000 resamples)
# =========================
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, 100 * (alpha/2))
    upper = np.percentile(boot_means, 100 * (1 - alpha/2))

    return lower, upper

# =========================
# Compute results
# =========================
results = []

for col in df.columns:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1)
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

print(result_df.to_string(index=False))

# save
result_df.to_excel("CNN_bootstrap_CI_5000.xlsx", index=False)

            Metric               Mean ± SD           Bootstrap 95% CI
      Val_Accuracy     0.978804 ± 0.002538       [0.977516, 0.980430]
          Val_Loss     0.065750 ± 0.006857       [0.061206, 0.069224]
               MCC     0.978120 ± 0.002619       [0.976828, 0.979775]
           ROC_AUC     0.999897 ± 0.000016       [0.999888, 0.999906]
       F1_Weighted     0.978807 ± 0.002533       [0.977584, 0.980520]
Precision_Weighted     0.979102 ± 0.002462       [0.977852, 0.980732]
   Recall_Weighted     0.978804 ± 0.002538       [0.977540, 0.980419]
     Training_Time 1105.943999 ± 81.880285 [1057.344497, 1154.464330]
       Memory_Used 451.601953 ± 306.935428   [332.425850, 656.256230]


In [ ]:
import pandas as pd
import numpy as np

# =========================
# MobilenetV2 with undersampling
# =========================
df = pd.DataFrame({
    "Val_Accuracy": [0.969966, 0.967009, 0.965297, 0.969032, 0.971366, 0.971366, 0.966698, 0.971366, 0.971833, 0.969655],
    "Val_Loss": [0.098850, 0.106081, 0.110569, 0.100131, 0.086578, 0.096508, 0.096945, 0.092812, 0.097009, 0.104773],
    "MCC": [0.969022, 0.965981, 0.964213, 0.968072, 0.970465, 0.970466, 0.965644, 0.970453, 0.970936, 0.968696],
    "ROC_AUC": [0.999781, 0.999778, 0.999757, 0.999802, 0.999819, 0.999781, 0.999813, 0.999754, 0.999782, 0.999734],
    "F1_Weighted": [0.969993, 0.966887, 0.965347, 0.969006, 0.971334, 0.971418, 0.966690, 0.971399, 0.971791, 0.969642],
    "Precision_Weighted": [0.970772, 0.967903, 0.966436, 0.970171, 0.971985, 0.972139, 0.967300, 0.971751, 0.972093, 0.970249],
    "Recall_Weighted": [0.969966, 0.967009, 0.965297, 0.969032, 0.971366, 0.971366, 0.966698, 0.971366, 0.971833, 0.969655],
    "Training_Time": [772.689698, 753.819622, 742.854679, 774.731109, 804.732023, 798.584231, 818.716810, 816.037896, 826.413140, 833.161924],
    "Memory_Used": [1325.367188, 493.367188, 372.617188, 361.617188, 366.523438, 371.375000, 375.937500, 435.292969, 323.941406, 320.781250]
})

# =========================
# Bootstrap CI (5000 resamples)
# =========================
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, 100 * (alpha/2))
    upper = np.percentile(boot_means, 100 * (1 - alpha/2))

    return lower, upper

# =========================
# Compute results
# =========================
results = []

for col in df.columns:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1)
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

print(result_df.to_string(index=False))

# save
result_df.to_excel("CNN_bootstrap_CI_5000.xlsx", index=False)

            Metric               Mean ± SD         Bootstrap 95% CI
      Val_Accuracy     0.969359 ± 0.002306     [0.967989, 0.970650]
          Val_Loss     0.099026 ± 0.006877     [0.095122, 0.103089]
               MCC     0.968395 ± 0.002374     [0.966988, 0.969724]
           ROC_AUC     0.999780 ± 0.000027     [0.999764, 0.999795]
       F1_Weighted     0.969351 ± 0.002313     [0.967950, 0.970670]
Precision_Weighted     0.970080 ± 0.002134     [0.968797, 0.971288]
   Recall_Weighted     0.969359 ± 0.002306     [0.967927, 0.970619]
     Training_Time  794.174113 ± 31.397933 [775.354006, 811.734625]
       Memory_Used 474.682031 ± 303.186055 [357.907373, 671.076631]


In [ ]:
import pandas as pd
import numpy as np

# =========================
# Data (10 runs) - Updated with MobileNetV2 results
# =========================
df = pd.DataFrame({
    "Val_Accuracy": [0.984600, 0.985411, 0.984510, 0.984150, 0.980457, 0.983429, 0.982529, 0.983429, 0.981178, 0.984420],
    "Val_Loss": [0.051388, 0.050550, 0.051002, 0.057354, 0.063336, 0.053651, 0.054259, 0.056249, 0.057809, 0.053254],
    "MCC": [0.984108, 0.984944, 0.984014, 0.983644, 0.979833, 0.982901, 0.981974, 0.982903, 0.980578, 0.983926],
    "ROC_AUC": [0.999900, 0.999890, 0.999895, 0.999846, 0.999834, 0.999895, 0.999918, 0.999856, 0.999888, 0.999901],
    "F1_Weighted": [0.984601, 0.985407, 0.984508, 0.984154, 0.980464, 0.983433, 0.982528, 0.983437, 0.981174, 0.984441],
    "Precision_Weighted": [0.984733, 0.985516, 0.984611, 0.984338, 0.980644, 0.983615, 0.982784, 0.983693, 0.981383, 0.984732],
    "Recall_Weighted": [0.984600, 0.985411, 0.984510, 0.984150, 0.980457, 0.983429, 0.982529, 0.983429, 0.981178, 0.984420],
    "Training_Time": [1264.833052, 1241.182733, 1312.565892, 1356.992223, 1377.847573, 1398.909364, 1406.495486, 1411.506954, 1442.305511, 1471.719643],
    "Memory_Used": [1248.578125, 424.230469, 297.898438, 296.703125, 382.175781, 311.750000, 260.507812, 249.738281, 349.574219, 290.585938]
})

# =========================
# Bootstrap CI (5000 resamples)
# =========================
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, 100 * (alpha/2))
    upper = np.percentile(boot_means, 100 * (1 - alpha/2))

    return lower, upper

# =========================
# Compute results
# =========================
results = []

for col in df.columns:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1)
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

print(result_df.to_string(index=False))

# save
result_df.to_excel("MobileNetV2_bootstrap_CI_5000.xlsx", index=False)

            Metric               Mean ± SD           Bootstrap 95% CI
      Val_Accuracy     0.983411 ± 0.001587       [0.982438, 0.984285]
          Val_Loss     0.054885 ± 0.003924       [0.052745, 0.057341]
               MCC     0.982882 ± 0.001638       [0.981860, 0.983811]
           ROC_AUC     0.999882 ± 0.000027       [0.999864, 0.999897]
       F1_Weighted     0.983415 ± 0.001588       [0.982432, 0.984295]
Precision_Weighted     0.983605 ± 0.001568       [0.982665, 0.984454]
   Recall_Weighted     0.983411 ± 0.001587       [0.982430, 0.984285]
     Training_Time 1368.435843 ± 74.999589 [1324.861900, 1411.678733]
       Memory_Used 411.174219 ± 299.105130   [294.192002, 607.170723]


In [1]:
import pandas as pd
import numpy as np

# تثبيت العشوائية لضمان تكرار المخرجات بدقة
np.random.seed(42)

# ==========================================
# CNN_LSTM with original dataset
# ==========================================
df = pd.DataFrame({
    "Val_Accuracy": [0.984399, 0.971919, 0.981279, 0.985959, 0.978159, 0.982839, 0.982839, 0.978159, 0.976599, 0.976599],
    "Val_Loss": [0.114087, 0.121744, 0.090742, 0.104071, 0.097027, 0.103456, 0.095808, 0.105955, 0.105546, 0.110877],
    "MCC": [0.983920, 0.971063, 0.980691, 0.985526, 0.977494, 0.982298, 0.982312, 0.977509, 0.975893, 0.975904],
    "ROC_AUC": [0.999581, 0.999695, 0.999743, 0.998721, 0.999677, 0.999252, 0.999538, 0.999559, 0.999400, 0.999255],
    "F1_Weighted": [0.984343, 0.971611, 0.981249, 0.985941, 0.978177, 0.982809, 0.982769, 0.978017, 0.976269, 0.976343],
    "Precision_Weighted": [0.985346, 0.973655, 0.982010, 0.986827, 0.979914, 0.983512, 0.983872, 0.980085, 0.978098, 0.978521],
    "Recall_Weighted": [0.984399, 0.971919, 0.981279, 0.985959, 0.978159, 0.982839, 0.982839, 0.978159, 0.976599, 0.976599],
    "Training_Time": [466.962099, 463.374990, 463.854516, 467.300552, 471.926163, 467.733991, 469.456476, 472.414571, 466.747848, 464.868309],
    "Memory_Used": [1036.382812, 246.851562, 292.625000, 174.617188, 383.781250, 255.605469, 261.843750, 357.714844, 298.648438, 360.863281]
})

# ==========================================
# دالة حساب فترة الثقة بنظام البوتستراب
# ==========================================
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, 100 * (alpha/2))
    upper = np.percentile(boot_means, 100 * (1 - alpha/2))

    return lower, upper

# ==========================================
# حلقة الحساب واستخراج المقاييس الإحصائية
# ==========================================
results = []

for col in df.columns:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1) # الانحراف المعياري للعينة
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

# عرض الجدول النهائي في سطر الأوامر
print(result_df.to_string(index=False))

# حفظ المخرجات في ملف إكسيل نهائي ومستقر
result_df.to_excel("MobileNetV2_Updated_Analysis_5000.xlsx", index=False)
print("\n[تمت العملية] تم حفظ النتائج بنجاح في ملف Excel.")

            Metric               Mean ± SD         Bootstrap 95% CI
      Val_Accuracy     0.979875 ± 0.004316     [0.977379, 0.982371]
          Val_Loss     0.104931 ± 0.009148     [0.099545, 0.110414]
               MCC     0.979261 ± 0.004442     [0.976532, 0.981831]
           ROC_AUC     0.999442 ± 0.000306     [0.999243, 0.999605]
       F1_Weighted     0.979753 ± 0.004417     [0.977041, 0.982289]
Precision_Weighted     0.981184 ± 0.003924     [0.978820, 0.983335]
   Recall_Weighted     0.979875 ± 0.004316     [0.977379, 0.982371]
     Training_Time   467.463952 ± 3.086519 [465.675582, 469.350147]
       Memory_Used 366.893359 ± 243.492110 [263.905654, 527.783828]

[تمت العملية] تم حفظ النتائج بنجاح في ملف Excel.


In [3]:
import pandas as pd
import numpy as np

# تثبيت العشوائية لضمان استقرار النتائج عند كل تشغيل
np.random.seed(42)

# ==========================================
# cnn_lstm with undersampling
# ==========================================
df = pd.DataFrame({
    "Val_Accuracy": [0.995851, 0.995851, 0.995851, 0.993776, 0.995851, 0.995851, 0.995851, 0.995851, 0.995851, 0.995851],
    "Val_Loss": [0.019020, 0.014143, 0.016820, 0.033855, 0.019125, 0.020543, 0.011627, 0.016832, 0.014921, 0.020702],
    "MCC": [0.995726, 0.995726, 0.995726, 0.993588, 0.995726, 0.995726, 0.995726, 0.995726, 0.995726, 0.995726],
    "ROC_AUC": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    "F1_Weighted": [0.995846, 0.995846, 0.995846, 0.993767, 0.995846, 0.995846, 0.995846, 0.995846, 0.995846, 0.995846],
    "Precision_Weighted": [0.996110, 0.996110, 0.996110, 0.994157, 0.996110, 0.996110, 0.996110, 0.996110, 0.996110, 0.996110],
    "Recall_Weighted": [0.995851, 0.995851, 0.995851, 0.993776, 0.995851, 0.995851, 0.995851, 0.995851, 0.995851, 0.995851],
    "Training_Time": [497.765136, 510.435865, 491.247386, 494.175308, 531.017695, 514.278084, 502.165158, 500.531753, 504.917428, 488.511259],
    "Memory_Used": [224.101562, 264.925781, 80.250000, 233.605469, 46.300781, 59.695312, 257.679688, 127.167969, 79.750000, 236.667969]
})

# ==========================================
# دالة البوتستراب لحساب فترات الثقة 95%
# ==========================================
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, 100 * (alpha/2))
    upper = np.percentile(boot_means, 100 * (1 - alpha/2))

    return lower, upper

# ==========================================
# إجراء الحسابات الإحصائية لجميع الأعمدة
# ==========================================
results = []

for col in df.columns:
    values = df[col]

    mean = np.mean(values)
    std = np.std(values, ddof=1) # الانحراف المعياري للعينة
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)

    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

# طباعة المخرجات على الشاشة
print(result_df.to_string(index=False))

# حفظ المخرجات في ملف إكسيل
result_df.to_excel("CNN_runs_statistical_analysis.xlsx", index=False)
print("\n[تم الحفظ] تم استخراج الإحصائيات لنموذج CNN بنجاح.")

            Metric              Mean ± SD         Bootstrap 95% CI
      Val_Accuracy    0.995644 ± 0.000656     [0.995228, 0.995851]
          Val_Loss    0.018759 ± 0.006050     [0.015740, 0.022606]
               MCC    0.995512 ± 0.000676     [0.995085, 0.995726]
           ROC_AUC    1.000000 ± 0.000000     [1.000000, 1.000000]
       F1_Weighted    0.995638 ± 0.000657     [0.995222, 0.995846]
Precision_Weighted    0.995915 ± 0.000618     [0.995524, 0.996110]
   Recall_Weighted    0.995644 ± 0.000656     [0.995228, 0.995851]
     Training_Time 503.504507 ± 12.599495 [496.772127, 511.630948]
       Memory_Used 161.014453 ± 89.945547 [107.920127, 212.409610]

[تم الحفظ] تم استخراج الإحصائيات لنموذج CNN بنجاح.


In [7]:
import pandas as pd
import numpy as np

# تثبيت العشوائية لضمان مطابقة النتائج عند كل تشغيل
np.random.seed(42)

# ==========================================
# إدخال الـ 10 تجارب بالقيم الأخيرة والكاملة بدقة
# ==========================================
df = pd.DataFrame({
    "Val_Accuracy": [0.985594, 0.977191, 0.986795, 0.986795, 0.987995, 0.987995, 0.986795, 0.983193, 0.986795, 0.985594],
    "Val_Loss": [0.086132, 0.114269, 0.086617, 0.075819, 0.092017, 0.084713, 0.092436, 0.088627, 0.090246, 0.098790],
    "MCC": [0.985151, 0.976504, 0.986386, 0.986385, 0.987623, 0.987626, 0.986394, 0.982688, 0.986386, 0.985159],
    "ROC_AUC": [0.998971, 0.998369, 0.998241, 0.998631, 0.998631, 0.998341, 0.997607, 0.998520, 0.998001, 0.998161],
    "F1_Weighted": [0.985534, 0.977107, 0.986807, 0.986787, 0.988011, 0.987941, 0.986777, 0.983221, 0.986764, 0.985593],
    "Precision_Weighted": [0.986160, 0.978457, 0.987342, 0.987268, 0.988470, 0.988430, 0.987517, 0.984339, 0.987277, 0.986436],
    "Recall_Weighted": [0.985594, 0.977191, 0.986795, 0.986795, 0.987995, 0.987995, 0.986795, 0.983193, 0.986795, 0.985594],
    "Training_Time": [835.727035, 836.597090, 934.836321, 853.189558, 934.550119, 858.733082, 859.930250, 907.257691, 948.909325, 911.263412],
    "Memory_Used": [339.402344, 241.265625, 75.707031, 89.765625, 86.687500, 74.023438, 188.757812, 22.035156, 14.000000, 212.347656]
})

# دالة حساب الـ Bootstrap CI لـ 5000 عينة
def bootstrap_ci(data, n_boot=5000, alpha=0.05):
    data = np.array(data)
    boot_means = []
    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))
    lower = np.percentile(boot_means, 100 * (alpha/2))
    upper = np.percentile(boot_means, 100 * (1 - alpha/2))
    return lower, upper

# تطبيق الحلقة الحسابية لجميع الأعمدة والمقاييس
results = []
for col in df.columns:
    values = df[col]
    mean = np.mean(values)
    std = np.std(values, ddof=1) # الانحراف المعياري للعينة
    ci_low, ci_high = bootstrap_ci(values, n_boot=5000)
    results.append({
        "Metric": col,
        "Mean ± SD": f"{mean:.6f} ± {std:.6f}",
        "Bootstrap 95% CI": f"[{ci_low:.6f}, {ci_high:.6f}]"
    })

result_df = pd.DataFrame(results)

# عرض المخرجات وحفظها في إكسيل
print(result_df.to_string(index=False))
result_df.to_excel("CNN_Runs_Final_Analysis_5000.xlsx", index=False)
print("\n[نجاح] تم حفظ وتحليل البيانات بدقة تامة.")

            Metric               Mean ± SD         Bootstrap 95% CI
      Val_Accuracy     0.985474 ± 0.003224     [0.983430, 0.987035]
          Val_Loss     0.090967 ± 0.010134     [0.085547, 0.097122]
               MCC     0.985030 ± 0.003317     [0.982807, 0.986635]
           ROC_AUC     0.998347 ± 0.000379     [0.998114, 0.998564]
       F1_Weighted     0.985454 ± 0.003240     [0.983283, 0.986919]
Precision_Weighted     0.986170 ± 0.002961     [0.984198, 0.987563]
   Recall_Weighted     0.985474 ± 0.003224     [0.983313, 0.987035]
     Training_Time  888.099388 ± 43.733281 [862.747508, 913.932767]
       Memory_Used 134.399219 ± 105.912833  [75.095019, 197.807900]

[نجاح] تم حفظ وتحليل البيانات بدقة تامة.
